<a href="https://colab.research.google.com/github/jazminfuentesb/ejercicios_ingenieria_en_datos_e_ia/blob/main/Ejercicios_06_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Resolución mediante el método simplex

In [5]:
#Bibliotecas
import numpy as np
from scipy.optimize import linprog
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [7]:
# ============================================================
# FUNCIONES BASE
# ============================================================
def simplex(c, A, b):
    """
    Implementa el algoritmo simplex para resolver problemas de programación lineal.

    Parámetros:
    c : vector de coeficientes de la función objetivo
    A : matriz de coeficientes de restricciones
    b : vector de recursos disponibles

    Retorna:
    solución óptima (valores de variables y Z)
    """

    num_vars = len(c)
    num_constraints = len(b)

    # Crear tabla simplex
    tableau = np.zeros((num_constraints + 1, num_vars + num_constraints + 1))

    # Llenar restricciones
    tableau[:-1, :num_vars] = A
    tableau[:-1, num_vars:num_vars + num_constraints] = np.eye(num_constraints)
    tableau[:-1, -1] = b

    # Función objetivo
    tableau[-1, :num_vars] = -c

    # Iteraciones del método simplex
    while any(tableau[-1, :-1] < 0):

        # Seleccionar columna pivote
        pivot_column = np.argmin(tableau[-1, :-1])

        # Calcular razones
        ratios = np.divide(
            tableau[:-1, -1],
            tableau[:-1, pivot_column],
            out=np.full_like(tableau[:-1, -1], np.inf),
            where=tableau[:-1, pivot_column] > 0
        )

        # Seleccionar fila pivote
        pivot_row = np.argmin(ratios)

        # Elemento pivote
        pivot_element = tableau[pivot_row, pivot_column]

        # Normalizar fila pivote
        tableau[pivot_row, :] /= pivot_element

        # Hacer ceros en la columna pivote
        for i in range(num_constraints + 1):
            if i != pivot_row:
                tableau[i, :] -= tableau[i, pivot_column] * tableau[pivot_row, :]

    return tableau


# ============================================================
# 🔷 FUNCIÓN PARA EXTRAER RESULTADOS
# ============================================================

def extract_solution(tableau, num_vars):
    """
    Extrae los valores de x1, x2,... y Z del tableau final
    """
    solution = np.zeros(num_vars)

    for i in range(num_vars):
        col = tableau[:, i]
        if list(col[:-1]).count(1) == 1 and list(col[:-1]).count(0) == len(col[:-1]) - 1:
            row = np.where(col[:-1] == 1)[0][0]
            solution[i] = tableau[row, -1]

    Z = tableau[-1, -1]

    return solution, Z




## Ejercicio dual 1

In [9]:
# ============================================
# EJERCICIO 1: Producción Industrial
# ============================================
# Contexto:
# Una empresa produce dos productos: A y B.
# Cada unidad de A genera una ganancia de $50
# Cada unidad de B genera una ganancia de $40.
#
# Recursos disponibles:
# Existen dos tipos de recursos limitados que restringen la producción.
#
# Restricciones:
# 1. Recurso tipo 1: máximo 100 unidades disponibles
#    - Producto A utiliza 3 unidades por cada unidad producida
#    - Producto B utiliza 2 unidades por cada unidad producida
#
# 2. Recurso tipo 2: máximo 90 unidades disponibles
#    - Producto A utiliza 2 unidades por cada unidad producida
#    - Producto B utiliza 3 unidades por cada unidad producida
#
# 3. No negatividad:
#    - No se pueden producir cantidades negativas
#
# Objetivo:
# Maximizar la ganancia total:
# Z = 50x1 + 40x2
#
# Restricciones matemáticas:
# 3x1 + 2x2 <= 100
# 2x1 + 3x2 <= 90
# x1, x2 >= 0
#
# ============================================
# FORMULACIÓN DUAL
# ============================================
# Problema dual asociado (minimización de recursos):
#
# Min Z = 100y1 + 90y2
#
# Restricciones:
# 3y1 + 2y2 >= 50   (correspondiente a x1)
# 2y1 + 3y2 >= 40   (correspondiente a x2)
#
# y1, y2 >= 0
#
# Para usarlo en el algoritmo simplex (forma estándar <=),
# multiplicamos las desigualdades por -1:
#
# -3y1 - 2y2 <= -50
# -2y1 - 3y2 <= -40
#
# ============================================
# VARIANTES PARA ANÁLISIS
# ============================================
#
# 🔹 Minimización:
# Minimizar costos de producción:
# Z = -50x1 - 40x2
#
# 🔹 Análisis de Sensibilidad:
# Variar los recursos disponibles:
# b1 = 100 ± Δ
# b2 = 90 ± Δ
#
# 🔹 Post-Optimización:
# Evaluar cambios en:
# - Coeficientes de la función objetivo (50, 40)
# - Restricciones (100, 90)
#
# ============================================

In [8]:
# Función objetivo: Z = 50x1 + 40x2
c = np.array([50, 40])

# Restricciones
A = np.array([
    [3, 2],
    [2, 3]
])

b = np.array([100, 90])


# ============================================================
# 🔷 MAXIMIZACIÓN
# ============================================================

print("🔹 MAXIMIZACIÓN")

tableau_max = simplex(c, A, b)
solution_max, Z_max = extract_solution(tableau_max, len(c))

print(f"Solución óptima: x1 = {solution_max[0]:.2f}, x2 = {solution_max[1]:.2f}")
print(f"Valor máximo de Z = {Z_max:.2f}")


# ============================================================
# 🔷 MINIMIZACIÓN
# ============================================================

print("\n🔹 MINIMIZACIÓN")

tableau_min = simplex(-c, A, b)
solution_min, Z_min = extract_solution(tableau_min, len(c))

print(f"Solución: x1 = {solution_min[0]:.2f}, x2 = {solution_min[1]:.2f}")
print(f"Valor mínimo de Z = {-Z_min:.2f}")


# ============================================================
# 🔷 ANÁLISIS DE SENSIBILIDAD
# ============================================================

print("\n🔹 ANÁLISIS DE SENSIBILIDAD")

for delta in range(-10, 11, 5):
    new_b = np.array([100 + delta, 90 + delta])

    tableau_sens = simplex(c, A, new_b)
    solution_sens, Z_sens = extract_solution(tableau_sens, len(c))

    print(f"\nCambio en recursos: {delta}")
    print(f"x1 = {solution_sens[0]:.2f}, x2 = {solution_sens[1]:.2f}, Z = {Z_sens:.2f}")


# ============================================================
# 🔷 POST-OPTIMIZACIÓN
# ============================================================

print("\n🔹 POST-OPTIMIZACIÓN")

original_solution = solution_max

for delta in range(-5, 6):
    new_b = b.copy()
    new_b[0] += delta

    tableau_post = simplex(c, A, new_b)
    solution_post, Z_post = extract_solution(tableau_post, len(c))

    print(f"\nCambio en b1: {delta}")
    print(f"x1 = {solution_post[0]:.2f}, x2 = {solution_post[1]:.2f}, Z = {Z_post:.2f}")

🔹 MAXIMIZACIÓN
Solución óptima: x1 = 24.00, x2 = 14.00
Valor máximo de Z = 1760.00

🔹 MINIMIZACIÓN
Solución: x1 = 0.00, x2 = 0.00
Valor mínimo de Z = -0.00

🔹 ANÁLISIS DE SENSIBILIDAD

Cambio en recursos: -10
x1 = 22.00, x2 = 12.00, Z = 1580.00

Cambio en recursos: -5
x1 = 23.00, x2 = 13.00, Z = 1670.00

Cambio en recursos: 0
x1 = 24.00, x2 = 14.00, Z = 1760.00

Cambio en recursos: 5
x1 = 25.00, x2 = 15.00, Z = 1850.00

Cambio en recursos: 10
x1 = 26.00, x2 = 16.00, Z = 1940.00

🔹 POST-OPTIMIZACIÓN

Cambio en b1: -5
x1 = 21.00, x2 = 16.00, Z = 1690.00

Cambio en b1: -4
x1 = 21.60, x2 = 15.60, Z = 1704.00

Cambio en b1: -3
x1 = 22.20, x2 = 15.20, Z = 1718.00

Cambio en b1: -2
x1 = 22.80, x2 = 14.80, Z = 1732.00

Cambio en b1: -1
x1 = 23.40, x2 = 14.40, Z = 1746.00

Cambio en b1: 0
x1 = 24.00, x2 = 14.00, Z = 1760.00

Cambio en b1: 1
x1 = 24.60, x2 = 13.60, Z = 1774.00

Cambio en b1: 2
x1 = 25.20, x2 = 13.20, Z = 1788.00

Cambio en b1: 3
x1 = 25.80, x2 = 12.80, Z = 1802.00

Cambio en b1:

## Ejercicio dual 2

In [10]:
# ============================================
# EJERCICIO 2: Planificación de Transporte
# ============================================
# Contexto:
# Una empresa debe transportar mercancía entre dos ciudades utilizando
# dos rutas diferentes. Cada ruta tiene un costo asociado y limitaciones
# de capacidad.
#
# Variables:
# x1 = cantidad transportada por la ruta 1
# x2 = cantidad transportada por la ruta 2
#
# Costos:
# - Ruta 1: $8 por unidad transportada
# - Ruta 2: $6 por unidad transportada
#
# Restricciones:
# 1. Demanda mínima:
#    Se deben transportar al menos 50 unidades en total
#    x1 + x2 >= 50
#
# 2. Capacidad de la ruta 1:
#    Máximo 40 unidades
#    x1 <= 40
#
# 3. Capacidad de la ruta 2:
#    Máximo 30 unidades
#    x2 <= 30
#
# 4. No negatividad:
#    x1, x2 >= 0
#
# Objetivo:
# Minimizar el costo total de transporte:
# Z = 8x1 + 6x2
#
# ============================================
# FORMULACIÓN MATEMÁTICA (forma estándar)
# ============================================
#
# Min Z = 8x1 + 6x2
#
# Restricciones:
# x1 + x2 >= 50
# x1 <= 40
# x2 <= 30
# x1, x2 >= 0
#
# Para usar simplex (forma <=), transformamos:
#
# -x1 - x2 <= -50
# x1 <= 40
# x2 <= 30
#
# ============================================
# FORMULACIÓN DUAL
# ============================================
#
# Max Z = 50y1 + 40y2 + 30y3
#
# Restricciones:
# -y1 + y2 <= 8   (correspondiente a x1)
# -y1 + y3 <= 6   (correspondiente a x2)
#
# y1, y2, y3 >= 0
#
# ============================================
# VARIANTES PARA ANÁLISIS
# ============================================
#
# 🔹 Maximización (dual del problema):
# Maximizar eficiencia del transporte:
# Z = -8x1 - 6x2
#
# 🔹 Análisis de Sensibilidad:
# Variar:
# - Demanda mínima (50 ± Δ)
# - Capacidades (40 y 30)
#
# 🔹 Post-Optimización:
# Evaluar cambios en:
# - Costos de transporte (8, 6)
# - Límites de capacidad
#
# ============================================

In [12]:
# Función objetivo: Z = 8x1 + 6x2 (minimización)
c = np.array([8, 6])

# Restricciones (forma <=)
A = np.array([
    [-1, -1],   # x1 + x2 >= 50 → -x1 - x2 <= -50
    [1, 0],     # x1 <= 40
    [0, 1]      # x2 <= 30
])

b = np.array([-50, 40, 30])


# ============================================================
# 🔷 MINIMIZACIÓN
# ============================================================

print("\n🔹 MINIMIZACIÓN")

tableau_min = simplex(-c, A, b)
solution_min, Z_min = extract_solution(tableau_min, len(c))

print(f"Solución óptima: x1 = {solution_min[0]:.2f}, x2 = {solution_min[1]:.2f}")
print(f"Valor mínimo de Z = {-Z_min:.2f}")


# ============================================================
# 🔷 MAXIMIZACIÓN (dual equivalente)
# ============================================================

print("\n🔹 MAXIMIZACIÓN")

tableau_max = simplex(c, A, b)
solution_max, Z_max = extract_solution(tableau_max, len(c))

print(f"Solución: x1 = {solution_max[0]:.2f}, x2 = {solution_max[1]:.2f}")
print(f"Valor de Z = {Z_max:.2f}")


# ============================================================
# 🔷 ANÁLISIS DE SENSIBILIDAD
# ============================================================

print("\n🔹 ANÁLISIS DE SENSIBILIDAD")

for delta in range(-10, 11, 5):
    new_b = np.array([-50 + delta, 40, 30])

    tableau_sens = simplex(-c, A, new_b)
    solution_sens, Z_sens = extract_solution(tableau_sens, len(c))

    print(f"\nCambio en demanda: {delta}")
    print(f"x1 = {solution_sens[0]:.2f}, x2 = {solution_sens[1]:.2f}, Z = {-Z_sens:.2f}")


# ============================================================
# 🔷 POST-OPTIMIZACIÓN
# ============================================================

print("\n🔹 POST-OPTIMIZACIÓN")

original_solution = solution_min

# Cambios en la primera restricción (demanda)
for delta in range(-5, 6):
    new_b = b.copy()
    new_b[0] += delta

    tableau_post = simplex(-c, A, new_b)
    solution_post, Z_post = extract_solution(tableau_post, len(c))

    print(f"\nCambio en demanda (b1): {delta}")
    print(f"x1 = {solution_post[0]:.2f}, x2 = {solution_post[1]:.2f}, Z = {-Z_post:.2f}")


# Cambios en costos (función objetivo)
print("\n🔹 CAMBIOS EN COSTOS")

for delta in range(-2, 3):
    new_c = np.array([8 + delta, 6])

    tableau_cost = simplex(-new_c, A, b)
    solution_cost, Z_cost = extract_solution(tableau_cost, len(new_c))

    print(f"\nCambio en costo de x1: {delta}")
    print(f"x1 = {solution_cost[0]:.2f}, x2 = {solution_cost[1]:.2f}, Z = {-Z_cost:.2f}")


🔹 MINIMIZACIÓN
Solución óptima: x1 = 0.00, x2 = 0.00
Valor mínimo de Z = -0.00

🔹 MAXIMIZACIÓN
Solución: x1 = 40.00, x2 = 30.00
Valor de Z = 500.00

🔹 ANÁLISIS DE SENSIBILIDAD

Cambio en demanda: -10
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda: -5
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda: 0
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda: 5
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda: 10
x1 = 0.00, x2 = 0.00, Z = -0.00

🔹 POST-OPTIMIZACIÓN

Cambio en demanda (b1): -5
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): -4
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): -3
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): -2
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): -1
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): 0
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): 1
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): 2
x1 = 0.00, x2 = 0.00, Z = -0.00

Cambio en demanda (b1): 3
x1 = 0.00, x2 = 0.00, Z 

## Ejercicio dual 3

In [13]:
# ============================================
# EJERCICIO 3: Asignación de Recursos
# ============================================
# Contexto:
# Una empresa debe asignar recursos a dos proyectos: P1 y P2.
# Cada proyecto genera un nivel de eficacia distinto y consume
# recursos limitados como tiempo y presupuesto.
#
# Variables:
# x1 = cantidad de recursos asignados al proyecto P1
# x2 = cantidad de recursos asignados al proyecto P2
#
# Beneficios:
# - Proyecto P1: 30 unidades de eficacia por unidad asignada
# - Proyecto P2: 20 unidades de eficacia por unidad asignada
#
# Restricciones:
# 1. Tiempo disponible: máximo 120 unidades
#    - P1 requiere 4 unidades de tiempo
#    - P2 requiere 3 unidades de tiempo
#
# 2. Presupuesto disponible: máximo 100 unidades
#    - P1 requiere 2 unidades de presupuesto
#    - P2 requiere 5 unidades de presupuesto
#
# 3. No negatividad:
#    x1, x2 >= 0
#
# Objetivo:
# Maximizar la eficacia total:
# Z = 30x1 + 20x2
#
# ============================================
# FORMULACIÓN MATEMÁTICA
# ============================================
#
# Max Z = 30x1 + 20x2
#
# Restricciones:
# 4x1 + 3x2 <= 120
# 2x1 + 5x2 <= 100
# x1, x2 >= 0
#
# ============================================
# FORMULACIÓN DUAL
# ============================================
#
# Min Z = 120y1 + 100y2
#
# Restricciones:
# 4y1 + 2y2 >= 30   (correspondiente a x1)
# 3y1 + 5y2 >= 20   (correspondiente a x2)
#
# y1, y2 >= 0
#
# Para simplex (forma <=):
# -4y1 - 2y2 <= -30
# -3y1 - 5y2 <= -20
#
# ============================================
# VARIANTES PARA ANÁLISIS
# ============================================
#
# 🔹 Minimización:
# Minimizar recursos no utilizados:
# Z = -30x1 - 20x2
#
# 🔹 Análisis de Sensibilidad:
# Variar:
# - Tiempo disponible (120 ± Δ)
# - Presupuesto (100 ± Δ)
#
# 🔹 Post-Optimización:
# Evaluar cambios en:
# - Beneficios (30, 20)
# - Restricciones del sistema
#
# ============================================

In [14]:
# Función objetivo: Z = 30x1 + 20x2
c = np.array([30, 20])

# Restricciones
A = np.array([
    [4, 3],   # 4x1 + 3x2 <= 120
    [2, 5]    # 2x1 + 5x2 <= 100
])

b = np.array([120, 100])


# ============================================================
# 🔷 MAXIMIZACIÓN
# ============================================================

print("\n🔹 MAXIMIZACIÓN")

tableau_max = simplex(c, A, b)
solution_max, Z_max = extract_solution(tableau_max, len(c))

print(f"Solución óptima: x1 = {solution_max[0]:.2f}, x2 = {solution_max[1]:.2f}")
print(f"Valor máximo de Z = {Z_max:.2f}")


# ============================================================
# 🔷 MINIMIZACIÓN
# ============================================================

print("\n🔹 MINIMIZACIÓN")

tableau_min = simplex(-c, A, b)
solution_min, Z_min = extract_solution(tableau_min, len(c))

print(f"Solución: x1 = {solution_min[0]:.2f}, x2 = {solution_min[1]:.2f}")
print(f"Valor mínimo de Z = {-Z_min:.2f}")


# ============================================================
# 🔷 ANÁLISIS DE SENSIBILIDAD
# ============================================================

print("\n🔹 ANÁLISIS DE SENSIBILIDAD")

for delta in range(-10, 11, 5):
    new_b = np.array([120 + delta, 100 + delta])

    tableau_sens = simplex(c, A, new_b)
    solution_sens, Z_sens = extract_solution(tableau_sens, len(c))

    print(f"\nCambio en recursos: {delta}")
    print(f"x1 = {solution_sens[0]:.2f}, x2 = {solution_sens[1]:.2f}, Z = {Z_sens:.2f}")


# ============================================================
# 🔷 POST-OPTIMIZACIÓN
# ============================================================

print("\n🔹 POST-OPTIMIZACIÓN")

original_solution = solution_max

# Cambios en la primera restricción (tiempo)
for delta in range(-5, 6):
    new_b = b.copy()
    new_b[0] += delta

    tableau_post = simplex(c, A, new_b)
    solution_post, Z_post = extract_solution(tableau_post, len(c))

    print(f"\nCambio en tiempo (b1): {delta}")
    print(f"x1 = {solution_post[0]:.2f}, x2 = {solution_post[1]:.2f}, Z = {Z_post:.2f}")


# Cambios en beneficios (función objetivo)
print("\n🔹 CAMBIOS EN BENEFICIOS")

for delta in range(-5, 6, 2):
    new_c = np.array([30 + delta, 20])

    tableau_cost = simplex(new_c, A, b)
    solution_cost, Z_cost = extract_solution(tableau_cost, len(new_c))

    print(f"\nCambio en beneficio de x1: {delta}")
    print(f"x1 = {solution_cost[0]:.2f}, x2 = {solution_cost[1]:.2f}, Z = {Z_cost:.2f}")


🔹 MAXIMIZACIÓN
Solución óptima: x1 = 30.00, x2 = 0.00
Valor máximo de Z = 900.00

🔹 MINIMIZACIÓN
Solución: x1 = 0.00, x2 = 0.00
Valor mínimo de Z = -0.00

🔹 ANÁLISIS DE SENSIBILIDAD

Cambio en recursos: -10
x1 = 27.50, x2 = 0.00, Z = 825.00

Cambio en recursos: -5
x1 = 28.75, x2 = 0.00, Z = 862.50

Cambio en recursos: 0
x1 = 30.00, x2 = 0.00, Z = 900.00

Cambio en recursos: 5
x1 = 31.25, x2 = 0.00, Z = 937.50

Cambio en recursos: 10
x1 = 32.50, x2 = 0.00, Z = 975.00

🔹 POST-OPTIMIZACIÓN

Cambio en tiempo (b1): -5
x1 = 28.75, x2 = 0.00, Z = 862.50

Cambio en tiempo (b1): -4
x1 = 29.00, x2 = 0.00, Z = 870.00

Cambio en tiempo (b1): -3
x1 = 29.25, x2 = 0.00, Z = 877.50

Cambio en tiempo (b1): -2
x1 = 29.50, x2 = 0.00, Z = 885.00

Cambio en tiempo (b1): -1
x1 = 29.75, x2 = 0.00, Z = 892.50

Cambio en tiempo (b1): 0
x1 = 30.00, x2 = 0.00, Z = 900.00

Cambio en tiempo (b1): 1
x1 = 30.25, x2 = 0.00, Z = 907.50

Cambio en tiempo (b1): 2
x1 = 30.50, x2 = 0.00, Z = 915.00

Cambio en tiempo (b1)